# Time Operators

Streaming data is infinite. Kafi Streams is an in-memory stream processor. Memory is never infinite.

So of course you need *time operators* that effectively clean up memory so that your Kafi Streams processing pipeline has constant, not ever-growing memory usage.

Freeing memory of timed out data is implemented as [expiry](#expiry) in Kafi Streams.

The time operators also allow you to implement the [*time windows*](#windows) as e.g. in Kafka Streams, such as [tumbling](#tumbling), [hopping](#hopping), [cumulative](#cumulative), [sliding](#sliding) and [session](#session) windows. And moreover, Kafi Streams enables you to build arbitrary [new types of time windows](#custom) as well.

## Overview

[Preparation](#prep)

* [Expiry](#expiry)
  * [expire()](#expire-operator)
* [Time windows in general](#windows)
  * [Time Windows = expire + group + aggregate](#expire_group_aggregate)
  * [Watermarks](#watermarks)
* [Tumbling windows](#tumbling)
  * [expire_tumbling()](#expire_tumbling-operator)
  * [group_by_agg_tumbling()](#group_by_agg_tumbling-operator)
  * [Walkthrough](#tumbling_walkthrough)
* [Hopping windows](#hopping)
  * [expire_hopping()](#expire_hopping-operator)
  * [group_by_agg_hopping()](#group_by_agg_hopping-operator)
  * [Walkthrough](#hopping_walkthrough)
* [Cumulative windows](#cumulative)
  * [expire_cumulative()](#expire_cumulative-operator)
  * [group_by_agg_cumulative()](#group_by_agg_cumulative-operator)
  * [Walkthrough](#cumulative_walkthrough)
* [Sliding windows](#sliding)
  * [expire_sliding()](#expire_sliding-operator)
  * [group_by_agg_sliding()](#group_by_agg_sliding-operator)
  * [Walkthrough](#sliding_walkthrough)
* [Session windows](#session)
  * [expire_session()](#expire_session-operator)
  * [group_by_agg_session()](#group_by_agg_session-operator)
  * [Walkthrough](#session_walkthrough.ipynb)
* [Triggers and Custom windows](#custom)
  * [Session/Threshold windows](#threshold)
  * [Walkthrough](#threshold_walkthrough)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [ ]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator, OrderGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()
order_generator = OrderGenerator()

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "sink"

#

def run(built_tn):
    sink_m_list = []
    for i in range(100):
        # 1. Generate new data.
        click_m_list = click_generator.generate(100)
        customer_m_list = customer_generator.generate(100)

        # 2. Push the new data to the topology + incrementally process the new data + get the resulting changes.
        sink_str_m_list_dict = built_tn.process({click_source_str: click_m_list, customer_source_str: customer_m_list})
        m_list = sink_str_m_list_dict[sink_str]

        # 3. Print out the size of the pydbsp state.
        sys.stdout.write(f"\rStep: {i + 1}, Memory: {built_tn.get_state_size() / 1024}KB")

        # 4. Add the changes to the output list.
        sink_m_list += m_list

    print()
    print(len(sink_m_list))
    print(sink_m_list[-10:])

def process(built_tn, customer_id, price, ts, w=1):
    m = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    #
    sink_str_r_list_dict = built_tn.process({order_source_str: [(m, w)]})
    r_list = sink_str_r_list_dict[sink_str]
    #
    print("Triggers:")
    for r in r_list:
        print(r)



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Please also note that when we re-use the same example over and over again to illustrate how the operators work, we always mark the important new parts as follows:
```python
    # <------------------------------>
    ...important new parts...
    # <------------------------------>
```

---
<a id="expiry"></a>
## Expiry

Expiry is the central concept in Kafi Streams for freeing memory of timed out data.

Thanks to pydbsp, Kafi Streams can implement expiry natively, without having to bolt on any kind of mechanism on top.

Essentially, expiry has to be defined only once for each source at the beginning of the Kafi Streams topology. All the stateful operators downstream do not need any special handling - they are automatically cleaned up by the expired records percolating through the topology, one by one.

We need an example. Let us recollect the example from the [Quickstart](../quickstart.ipynb) using the `TopologyNode` class (see [Architecture](../architecture.ipynb))


In [ ]:
click_source_str = "clicks"
customer_source_str = "customers"

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
    .sink(sink_str)
)

built_tn = Tn.build(sink_tn)


And then, let us throw data at it and see how the global state size of the topology grows:

In [ ]:
built_tn.reset()
for _ in range(3):
    run(built_tn)

This is of course not sustainable. The memory usage grows unboundedly and Kafi Streams gets slower and slower (in this case, mostly caused by the join).

It's time to introduce the `expire()` operator.

<a id="expire-operator"></a>
### expire()

Expires individual records so that they can be purged from memory.

```
expire(ts_fun, expiry_fun, project_fun=lambda r_ts_tuple: r_ts_tuple[0], **kwargs)
```
* `ts_fun: r -> ts`: timestamp function - gets an input record and returns a timestamp
* `expiry_fun: ts -> ts`: expiry function - gets a timestamp and returns the corresponding expiry timestamp specifying when the record shall expire
* `project_fun: tuple(r, ts) -> r`: projection function - gets a pair of a record and its expiry timestamp and returns another record. Default: `lambda r_ts_tuple: r_ts_tuple[0]`

Let's add this to our topology.


In [ ]:
click_source_str = "clicks"
customer_source_str = "customers"

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    # <------------------------------>
    .expire(ts_fun=lambda r: r["ts"],
            expiry_fun=lambda ts: ts + click_generator.ts_step_int * 1000)
    # <------------------------------>
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
    .sink(sink_str)
)

built_tn = Tn.build(sink_tn)


Here, we use the `expiry()` operator to:
1. Select the `ts` field from each record,
2. and then set the expiry to the selected timestamp plus `1000` times the timestamp step size of the click generator.

When you look closer at the topology, you can observe multiple some of the properties of expiry in Kafi Streams:
* It is defined once at the top of the topology, before any stateful operator (`map` and `filter` are stateless).
* The stateful operators (`distinct` and `join_equi`) below in the topology do not need to know anything about the expiry.

Ok. Let's see this in action. Will be able to rein in the memory consumption?

In [ ]:
for _ in range(5):
    run(built_tn)

It works! Constant, flat memory usage! No processing slowdown!

Why? Because we used `expire()` to time out the transactional data (=the clicks). After a short while, the memory consumption of the master data (=the customers) becomes constant as well because it is limited (the generator only generates up to 100 customers in the example).

---
<a id="windows"></a>
## Time windows

In the previous section, we learnt how we can keep Kafi Streams' memory usage at check. It was only remotely related to time windows in the classical stream processing sense: under the covers, the `expire()` operator works akin to a "sliding window" in classical stream processing.

This section is about "real" stream processing time windows.

You'll see that we devised a novel formulation of them inside DBSP that allows us to build all the time window types from classical stream processing.

But it doesn't stop there - Kafi Streams is so flexible that you can easily build your own custom time windows.


<a id="expire_group_aggregate"></a>
### Time Windows = expire + group + aggregate

What is a time window really? You can think of time windows in stream processing as consisting of two ingredients:
* **Expiry**: Time windows have a start and an end. Events *expire* after the end of a time window so that they can be cleaned up.
* **Group By + Aggregate**: The actual "time window" is a set of events *grouped* by time and some other key (e.g. a customer ID) and *aggregated*.

Now this is very theoretical. Let's pick the simplest time window - the *tumbling window* and see how all this theory plays out in practice.

In [ ]:
# <------------------------------>

def ts_fun(r):
    return r["ts"]

size_int = click_generator.ts_step_int * 1000

# <------------------------------>

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    
    # <------------------------------>
    
    .expire_tumbling(ts_fun=ts_fun,
                     size_int=size_int)
    
    # <------------------------------>
    
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

joined_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]})
)

# <------------------------------>

sink_tn = (
    joined_tn
    .group_by_agg_tumbling(
        ts_fun=ts_fun,
        size_int=size_int,
        key_fun=lambda r: {"customer_id": r["customer_id"], "name": r["name"]},
        agg_fun=lambda agg_r, r: {"clicks": agg_r["clicks"] + 1,
                                  "view_times": agg_r["view_times"] + [r["view_time"]],
                                  "total_view_time": agg_r["total_view_time"] + r["view_time"]},
        agg_initial_any={"clicks": 0, "view_times": [], "total_view_time": 0},
        project_fun=lambda key_any, agg_r: {"customer_id": key_any["customer_id"],
                                            "name": key_any["name"],
                                            "clicks": agg_r["clicks"],
                                            "view_times": agg_r["view_times"],
                                            "total_view_time": agg_r["total_view_time"]})
    .sink(sink_str)
)

# <------------------------------>

built_tn = Tn.build(sink_tn)


What do we do here?

* **expire**: At the top of the topology, we specify the expiry of the incoming clicks using the `expire_tumbling()` operator and set the tumbling window size to `tumbling_size_int = click_generator.ts_step_int * 1000`.
* **group + aggregate**: After the join of clicks and customers, we create the tumbling window using the `group_by_agg_tumbling()` operator: We group by `customer_id` and `name`, and aggregate the clicks for that customer in that time window:
  * `clicks` the number of clicks of the customer
  * `view_times` the list of view times of the customer
  * `total_view_time` the total view time of the customer

Let's run this.

In [ ]:
run(built_tn)

This was your first time window in Kafi Streams in action!

In the following sections, we double down on the individual built-in window types of Kafi Streams and explain their API in detail.

<a id="watermarks"></a>
### Watermarks

In this version, Kafi Streams it does *not* use implicit *watermarks* for tracking the progress of time throughout the topology. The approximation of a watermark in Kafi Streams is the maximum timestamp of the messages that it has seen so far - in the following we refer to this as the *latest timestamp*.

This is a simplification and also clearly a trade-off. It makes the architecture of Kafi Streams simpler, but without implicit watermarking, Kafi Streams can drop time windows in case the gaps between the source events are bigger than the respective window size + allowed lateness.

There are workarounds for this, but, as not using implicit watermarks is a trade-off, none of them comes without downsides. Here are two:
* Increase the allowed lateness, with the disadvantage of higher memory consumption.
* Augment the source events with events that serve as "explicit watermarks", with the disadvantege of handing over this part of the complexity to the producer (e.g. dummy events that you put in regular intervals less than than window size + allowed lateness).


---
<a id="tumbling"></a>
## Tumbling Windows

In Kafi Streams, the two operators required to set up a tumbling window are `expire_tumbling` and `group_by_agg_tumbling`.

<a id="expire_tumbling-operator"></a>
### expire_tumbling()

Syntactic sugar for `expire()` for record expiry in the context of tumbling windows.

```
expire_tumbling(ts_fun, size_int, allowed_lateness_int=0, **kwargs)
```
* `ts_fun: r -> ts` timestamp function - gets an input record and returns a timestamp
* `size_int` the size (in milliseconds) of the tumbling window
* `allowed_lateness_int` allowed lateness - adds a time (in milliseconds) to the expiry time to accommodate late arriving records

Using this operator, the expiry time for a record with timestamp `ts` is: `(ts // size_int) * size_int + size_int + size_int + allowed_lateness_int`. Note that a second `size_int` needs to be added to avoid the record being discarded too early. We call this additional addition *window buffer time*.


<a id="group_by_agg_tumbling-operator"></a>
### group_by_agg_tumbling()

Augmented `group_by_agg()` operator for creating tumbling time windows by grouping by + aggregating. Implicitly also groups by the tumbling time windows and triggers the emission of aggregated tumbling time windows.

```
group_by_agg_tumbling(ts_fun, size_int, key_fun, agg_fun, agg_initial_any, project_fun, trigger_fun=lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1], trigger_project_fun=lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]}, trigger_positive_only_bool=True, **kwargs)
```

* `ts_fun: r -> ts`: timestamp function - gets an input record and returns a timestamp
* `size_int`: the size (in milliseconds) of the window
* `key_fun: r -> any`: the selection function for the key for the grouping
* `agg_fun: agg_any, value_any -> any`: the aggregation function; gets the result of the aggregation so far (`agg_any`) and the next selected value to aggregate (`value_any`), and returns the updated aggregation
* `agg_initial_any`: the initial value of the aggregation
* `project_fun: key_any, agg_any -> r`: the projection function; gets both the selected key and the aggregation result for that key and returns the output (=projection) of the group by + aggregation
* `trigger_fun: tuple(r, end_ts), latest_ts -> bool` the trigger function; gets a pair of a record and the window end timestamp and the latest timestamp of the input stream and returns a bool. `True` for triggering the emission of the output, `False` for not yet triggering it. Default: `lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1]` where `r_end_ts_tuple[1] = end_ts`- i.e., trigger if the latest timestamp of the input stream is greater than or equal to the window end timestamp `end_ts`.
* `trigger_project_fun: tuple(r, end_ts) -> r`: the trigger projection function; gets a pair of a record and the window end timestamp and returns a record. Default: `lambda r_end_ts_tuple: {**r_end_ts_int_tuple[0], "window_end": r_end_ts_int_tuple[1]}` i.e., add the window end timestamp to the record.
* `trigger_positive_only_bool`: trigger only updates with positive weights or also zero or negative ones. Default: `True`

This looks scary at first. But for most use cases, only these parameters are obligatory:
* `ts_fun` - same as in `expire_tumbling()`
* `size_int` - same as in `expire_tumbling()`
* `key_fun` - as in `group_by_agg()`
* `agg_fun` - as in `group_by_agg()`
* `agg_initial_any` - as in `group_by_agg()`
* `project_fun` - as in `group_by_agg()`

The `trigger_` parameters should only be required for advanced use cases. They control the emission of time windows based on a triggering mechanism. Their defaults should suffice for most uses cases. We'll show a use case for them when we discuss custom time windows at the end of this notebook.

Note that contrary to the basic `group_by_agg()` operator, there is no `value_fun`. This is because the `value_fun` in `group_by_agg_tumbling` is always the identity function since we assume that in 99% of the use cases, you would want to create time windows around entire records.


<a id="tumbling_walkthrough"></a>
### Walkthrough

Next, you can go through an exhaustive illustration of how Kafi Streams' tumbling window works by walking through some example data - one by one, in baby steps, graphically: [tumbling window walkthrough](windows/tumbling.ipynb)
